# 04 — Graph Generation and Evolutionary Optimization

**Goal:** Generate candidate graphs using classical generators, compute
structural metrics, then optimize a graph with evolutionary search toward
a connectivity target.

**TGraphX subsystem:** `tgraphx.generation`, `tgraphx.evolutionary`

**Data:** Synthetic — no download.

**Runtime:** < 60 seconds on CPU.

In [ ]:
from tgraphx import run_graph_generation, run_evolutionary_optimization
from tgraphx import list_graph_generation_methods, list_evolutionary_optimizers
print("Generation methods:", list(list_graph_generation_methods().keys()))
print("Evolutionary optimizers:", list(list_evolutionary_optimizers().keys()))

## 2. Generate Graphs with Structural Metrics

In [ ]:
# Generate 10 Erdős–Rényi and 10 Barabási–Albert graphs.
er_result = run_graph_generation(
    method="erdos_renyi", num_graphs=10, num_nodes=20,
    num_edges=40, seed=42,
)
ba_result = run_graph_generation(
    method="barabasi_albert", num_graphs=10, num_nodes=20,
    num_edges=40, seed=42,
)
for name, res in [("ER", er_result), ("BA", ba_result)]:
    m = res.metrics
    print(f"{name}: validity={m.get('validity',0):.2f}  "
          f"uniqueness={m.get('uniqueness',0):.2f}  "
          f"diversity={m.get('diversity',0):.3f}")

## 3. Evolve a Graph Toward High Connectivity

In [ ]:
from tgraphx.evolutionary import GraphGenome, GeneticAlgorithmOptimizer, GeneticAlgorithmConfig
from tgraphx.evolutionary import connectivity_fitness
import torch

def make_genome(seed=0):
    torch.manual_seed(seed)
    ei = torch.randint(0, 10, (2, 12))
    return GraphGenome(edge_index=ei, num_nodes=10)

config = GeneticAlgorithmConfig(population_size=12, n_generations=20, seed=42)
result = GeneticAlgorithmOptimizer(config, connectivity_fitness).optimize(
    [make_genome(i) for i in range(12)]
)
print(f"Best connectivity fitness: {result.best_fitness:.4f}")
print(f"Generations run: {len(result.history)}")

## 4. Why This Matters

Graph generation + evolution lets you:
- create synthetic graphs with controlled properties for benchmarking;
- apply multi-objective Pareto search when multiple metrics compete;
- search over graph structure jointly with model training;
- study graph property distributions.

TGraphX evolutionary utilities preserve tensor node features through
mutation/crossover if the genome carries them.

## 5. Next Steps
- **Tutorial:** `tutorials/graph_generation_quickstart.py`
- **NSGA-II multi-objective:** see `NSGAIIOptimizer`
- **Limitations:** these generators produce simple random graphs.
  Neural generation (VAE, autoregressive) is in `tgraphx.generation` but
  marked Experimental.